## In this notebook:

#### We concatenate parcellated PET images into region x receptor matrix of densities.
Adapted from Hansen Receptors (https://github.com/netneurolab/hansen_receptors)
Paper: https://www.nature.com/articles/s41593-022-01186-3

### Import Packages

In [1]:
import numpy as np
import pandas as pd
from netneurotools import datasets, plotting
from matplotlib.colors import ListedColormap
from scipy.stats import zscore
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_cammoun2012

In [2]:
import surfer

### Set Variables

In [3]:
path = '/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/'
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'
figpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/"
#'/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/figures/'

outpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/"
atlas = 'desikan'

if atlas == 'schaefer':
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17) # This is sklearn.utils._bunch.Bunch. Other altases in this format? *Ask Golia
    nnodes = len(schaefer['labels'])
elif atlas == 'cammoun':
    scale = 'cammoun2012_scale500' #'scale1000_17' #'Harvard_Oxford_sub' 
    folder = 'atl-Cammoun2012_res-500'
    cammoun = fetch_cammoun2012()
    nnodes=1015
elif atlas == 'Harvard_Oxford_sub':
    nnodes = 21
elif atlas == 'desikan':
    nnodes = 112
    scale = 'desikan_killiany'
    desikan = '~/Desktop/desikan-killiany_aparcaseg.nii.gz'
    #desikan = '/Users/pecsok/nilearn_data/desikan_killiany/desikan-killiany2006_rois_1mm.nii.gz'
    atlas = 'desikan'
    parcel_file_path = '/Users/pecsok/nilearn_data/desikan_killiany/desikanKillianyNodeNames.txt'
    labels = pd.read_csv(parcel_file_path, header=None, names=['ParcelName'])


In [4]:
pd.set_option('display.max_rows', None)
#print(labels)

cortical = pd.read_csv("~/Desktop/cortical_significance_analysis.csv", sep=',')
subcortical = pd.read_csv("~/Desktop/subcortical_significance_analysis.csv", sep=',')


### Choose what to analyse

In [113]:
receptors_csv = [path+'data/parcellated/PET_parcellated/'+scale+'/5HT1a_way_hc36_savli.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/5HT1b_p943_hc22_savli.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/5HT1b_p943_hc65_gallezot.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/5HT2a_cimbi_hc29_beliveau.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/5HT4_sb20_hc59_beliveau.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/5HT6_gsk_hc30_radhakrishnan.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/5HTT_dasb_hc100_beliveau.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/A4B2_flubatine_hc30_hillmer.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/CB1_omar_hc77_normandin.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/D1_SCH23390_hc13_kaller.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/D2_flb457_hc37_smith.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/D2_flb457_hc55_sandiego.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/DAT_fpcit_hc174_dukart_spect.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/GABAa_flumazenil_hc6_dukart.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/H3_cban_hc8_gallezot.csv', 
                 path+'data/parcellated/PET_parcellated/'+scale+'/M1_lsn_hc24_naganawa.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/mGluR5_abp_hc22_rosaneto.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/mGluR5_abp_hc28_dubois.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/mGluR5_abp_hc73_smart.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/MU_carfentanil_hc204_kantonen.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/NAT_MRB_hc77_ding.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/NMDA_ge179_hc29_galovic.csv',
                 #path+'data/parcellated/PET_parcellated/'+scale+'/VAChT_feobv_hc3_spreng.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/VAChT_feobv_hc4_tuominen.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/VAChT_feobv_hc5_bedard_sum.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'/VAChT_feobv_hc18_aghourian_sum.csv']



subcortical_significance_analysis.csv

"""
receptors_csv = [path+'data/parcellated/PET_parcellated/'+scale+'/NMDA_ge179_hc29_galovic.csv',
                 path+'data/parcellated/PET_parcellated/'+scale+'GABAa-bz_flumazenil_hc16_norgaard.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'GABAa_flumazenil_hc6_dukart.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'mGluR5_abp_hc22_rosaneto.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'mGluR5_abp_hc28_dubois.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'mGluR5_abp_hc73_smart.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'NMDA_ge179_hc29_galovic.nii.gz',
                 path+'data/parcellated/PET_parcellated/'+scale+'DAT_fpcit_hc174_dukart_spect.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'DAT_fepe2i_hc6_sasaki.nii.gz',
                 path+'data/parcellated/PET_parcellated/'+scale+'CB1_omar_hc77_normandin.nii.gz',
                 path+'data/parcellated/PET_parcellated/'+scale+'CB1_FMPEPd2_hc22_laurikainen.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'VAChT_feobv_hc3_spreng.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'VAChT_feobv_hc4_tuominen.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'VAChT_feobv_hc5_bedard_sum.nii',
                 path+'data/parcellated/PET_parcellated/'+scale+'VAChT_feobv_hc18_aghourian_sum.nii']
"""

NameError: name 'subcortical_significance_analysis' is not defined

### Make Receptor Matrices

In [6]:
print(scale) #pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/"+scale+"

desikan_killiany


In [6]:
#print(receptors_csv)
pd.set_option('display.max_rows', None)

In [65]:
#print(labels)

In [82]:
# Create annoying loop to reframe df to make equivalent of receptor_df

voldf = pd.DataFrame(index=labels['ParcelName'], columns=['Hemisphere','Volume_change','Pvalue']).fillna(0)

#print(voldf)
for i, roi in enumerate(cortical['ROI']):
    parts = roi.split('_')
    hemi = parts[0]
    string = parts[1]
    for j in range(len(voldf)):
        region = str(voldf.index[j])
        
        # Check if the extracted string is in the index of voldf
        if string.lower() in region.lower() and hemi.lower() in region.lower():
            # Add the corresponding volume to voldf
            value = cortical.loc[i, 'Estimate']
            p = cortical.loc[i, 'P.Value']
            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
            voldf.at[region, 'Hemisphere'] = hemi
            voldf.at[region, 'Pvalue'] = p

# Subcortical
for i, roi in enumerate(subcortical['ROI']):
    parts = roi.split('.')
    hemi = parts[0]
    string = parts[1]
    print(string)
    for j in range(len(voldf)):
        region = str(voldf.index[j])
        
        # Check if the extracted string is in the index of voldf
        if string.lower() in region.lower() and hemi.lower() in region.lower():
            # Add the corresponding volume to voldf
            value = subcortical.loc[i, 'Estimate']
            p = cortical.loc[i, 'P.Value']
            voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
            voldf.at[region, 'Hemisphere'] = hemi
            voldf.at[region, 'Pvalue'] = p
            
# Display the resulting DataFrame
display(voldf)


Cerebellum
Thalamus
Caudate
Putamen
Pallidum
Hippocampus
Amygdala
Accumbens
VentralDC
Cerebellum
Thalamus
Caudate
Putamen
Pallidum
Hippocampus
Amygdala
Accumbens
VentralDC
Posterior
Mid_Posterior
Central
Mid_Anterior
Anterior


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_48676/28971284.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  voldf = pd.DataFrame(index=labels['ParcelName'], columns=['Hemisphere','Volume_change','Pvalue']).fillna(0)
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_48676/28971284.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '627.2975405' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  voldf.at[region, 'Volume_change'] = value  # Use 'at' for fast scalar access
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_48676/28971284.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will 

,Hemisphere,Volume_change,Pvalue
ParcelName,,,
Left-Cerebral-White-Matter,0,0.000000,0.000000e+00
Left-Lateral-Ventricle,0,0.000000,0.000000e+00
Left-Inf-Lat-Vent,0,0.000000,0.000000e+00
Left-Cerebellum-White-Matter,Left,6680.831629,8.480000e-10
Left-Cerebellum-Cortex,Left,6680.831629,8.480000e-10
Left-Thalamus-Proper,Left,1673.816025,2.390010e-04
Left-Caudate,Left,585.087394,5.790000e-12
Left-Putamen,Left,1209.638209,8.720000e-07
Left-Pallidum,Left,262.890897,1.720425e-01


In [90]:
# combine all the receptors (including repeats)
r = np.zeros([nnodes, len(receptors_csv)])
print(r.shape)
for i in range(len(receptors_csv)):
    r[:, i] = np.genfromtxt(receptors_csv[i], delimiter=',') 
    # Now we have the receptor data by parcel (row) for all 6 receptor maps (col)
    

#print(r[:,3])
print(r.shape)

receptor_names = np.array(["5HT1a", "5HT1b", "5HT2a", "5HT4", "5HT6", "5HTT", "A4B2",
                           "CB1", "D1", "D2", "DAT", "GABAa", "H3", "M1", "mGluR5",
                           "MOR", "NET", "NMDA", "VAChT"])
#np.save(path+'data/receptor_names_pet.npy', receptor_names)

# make final region x receptor matrix
receptor_data = np.zeros([nnodes, len(receptor_names)])

# Add 5HT1a Data
receptor_data[:, 0] = zscore(r[:, 0]) 
# Add "5HT2a", "5HT4", "5HT6", "5HTT", "A4B2", "CB1", "D1"
receptor_data[:, 2:9] = zscore(r[:, 3:10])
# Add "DAT", "GABAa", "H3", "M1"
receptor_data[:, 10:14] = zscore(r[:, 12:16])
# Add "MOR", "NET", "NMDA"
receptor_data[:, 15:18] = zscore(r[:, 19:22])

# weighted average of 5HT1B p943
receptor_data[:, 1] = (zscore(r[:, 1])*22 + zscore(r[:, 2])*65) / (22+65)
# weighted average of D2 flb457
receptor_data[:, 9] = (zscore(r[:, 10])*37 + zscore(r[:, 11])*55) / (37+55)
# weighted average of mGluR5 ABP688
receptor_data[:, 14] = (zscore(r[:, 16])*22 + zscore(r[:, 17])*28 + zscore(r[:, 18])*73) / (22+28+73)
# weighted average of VAChT FEOBV
receptor_data[:, 18] = (zscore(r[:, 22])*4 + zscore(r[:, 23])*5 + zscore(r[:, 24])*18) / (4+5+18)

print(receptors_csv[0]) #5HT1a

# Make df and add parcel labels
receptor_df = pd.DataFrame(receptor_data, columns=receptor_names,index=labels)

"""
#schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
#labels = schaefer.labels
#labels = [label.decode('utf-8') for label in labels]
cam = fetch_cammoun2012()
info = pd.read_csv(cammoun['info'], sep=',')
info = info[info['scale']== 'scale500']
hemisphere = info['hemisphere'].values
labels = info['label'].values + hemisphere
print(labels)
"""

receptor_df.index = labels
receptor_df.index.name = 'Parcel'

#print(receptor_df)
# Save df
receptor_df.to_csv(path+'results/receptor_data_'+scale+'.csv', index=True, header=True)



(112, 25)
(112, 25)
/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/5HT1a_way_hc36_savli.csv


In [93]:
#print(voldf)

In [98]:
# Make vol change image
r = np.zeros([nnodes, 1])
print(r.shape)
r[:,0] = voldf['Volume_change']

vol_data = r

(112, 1)


In [84]:
"""
plot receptor data
"""

# colourmaps
cmap = np.genfromtxt(datapath+'data/colourmap.csv', delimiter=',')
cmap_div = ListedColormap(cmap)

# plot each receptor map
if scale == 'scale1000_17':
    annot = datasets.fetch_schaefer2018('fsaverage')['1000Parcels17Networks']
    type(annot)
    print(annot)
    
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot=annot.lh,
                                        rhannot=annot.rh,
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(path+'figures/scale1000_17/surface_receptor_'+receptor_names[k]+'.png')


if scale == 'scale100':
    annot = datasets.fetch_schaefer2018('fsaverage')['100Parcels7Networks']
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot=annot.lh,
                                        rhannot=annot.rh,
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(path+'figures/' + scale + '/surface_receptor_'+receptor_names[k]+'.png')

## NEWWWWWW
if scale == 'cammoun2012_scale500':
    cammoun = datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    print(cammoun)
    #annot = datasets.fetch_schaefer2018('fsaverage')['100Parcels7Networks']
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot = cammoun[0],
                                        rhannot = cammoun[1],
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(figpath + folder + '/surface_receptor_'+receptor_names[k]+'.png')




#receptor_data[:, 2:9] = r[:, 3:10]
print(receptor_data)
print(len(receptor_data))

print(r)


Surface(lh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-L_deterministic.annot', rh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-R_deterministic.annot')


IndexError: index 146 is out of bounds for axis 0 with size 114

In [131]:
#if scale == 'cammoun2012_scale500':
cammoun = datasets.fetch_cammoun2012(version='fsaverage')
cammoun = cammoun['scale060']
print(cammoun)


Surface(lh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-060_hemi-L_deterministic.annot', rh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-060_hemi-R_deterministic.annot')


In [132]:
fsavg = transforms.mni152_to_fsaverage(desikan, '164k')
lhannot = fsavg[0]
ldata_array = lhannot.darrays[0].data
lunique_labels = np.unique(ldata_array)
llabels = [f'label_{int(label)}' for label in lunique_labels]
# Create a colormap with random RGB values for each unique label (or define your own)
lcolormap = np.zeros((len(lunique_labels) + 1, 4), dtype=np.int32)  # +1 for background
for i, label in enumerate(lunique_labels):
    colormap[i + 1] = [i * 40 % 255, i * 80 % 255, i * 120 % 255, 255]  # RGB + alpha

nib.freesurfer.write_annot(
    '/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-L_deterministic.annot',
    labels=ldata_array.astype(int),  # Label array from Gifti data
    ctab=lcolormap,                  # Color table
    names=llabels                    # List of region names
)  

# Right hemisphere
rhannot = fsavg[1]
rdata_array = rhannot.darrays[0].data
runique_labels = np.unique(rdata_array)
rlabels = [f'label_{int(label)}' for label in runique_labels]
# Create a colormap with random RGB values for each unique label (or define your own)
rcolormap = np.zeros((len(runique_labels) + 1, 4), dtype=np.int32)  # +1 for background
for i, label in enumerate(runique_labels):
    colormap[i + 1] = [i * 40 % 255, i * 80 % 255, i * 120 % 255, 255]  # RGB + alpha

nib.freesurfer.write_annot(
    '/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-R_deterministic.annot',
    labels=rdata_array.astype(int),  # Label array from Gifti data
    ctab=rcolormap,                  # Color table
    names=rlabels                    # List of region names
)

In [129]:
from neuromaps.datasets import fetch_annotation
from neuromaps import transforms
import nibabel as nib

lannot = '/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-L_deterministic.annot'
rannot = '/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-R_deterministic.annot'

if scale == 'desikan_killiany':    
    brain = plotting.plot_fsaverage(data=vol_data[:, 0],
                                    lhannot = lannot,
                                    rhannot = rannot,
                                    colormap='plasma',
                                    views=['lat', 'med'],
                                    data_kws={'representation': "wireframe"})
    brain.save_image(figpath + folder + '/surface_receptor_'+receptor_names[k]+'.png')



/Users/pecsok/Desktop/atl-desikan_space-fsaverage_res-112_hemi-L_deterministic.annot


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
info = pd.read_csv(cammoun['info'], sep=',')
info = info[info['scale']== 'scale500']
hemisphere = info['hemisphere'].values
labels = info['label'].values + hemisphere


